# PhyloP Conservation Analysis v3 - Exonic Flanking Regions

**Key improvement over v2:**
- v2: Genomic flanks (before first exon, after last exon) - includes introns
- v3: Exonic flanks (walks along transcript exons, skips introns)

This ensures flanks don't have artificially low conservation from intronic sequence.

## Setup and Configuration

In [ ]:
import sys
import pandas as pd
import numpy as np
import pyBigWig
import gzip
from collections import defaultdict
from pathlib import Path

# Install pyBigWig if needed
try:
    import pyBigWig
except ImportError:
    print("Installing pyBigWig...")
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'pyBigWig'])
    import pyBigWig
    print("Installation complete. Please restart kernel if needed.")

In [ ]:
# Configuration
V2_RESULTS = '../results/phylop/translon_phylop_v2_comprehensive.tsv'
TRANSCRIPT_ANNOTATIONS = '../../phase1_w_transcript.tsv'
GENCODE_GTF = '../../data/gencode.v46.annotation.gtf.gz'
PHYLOP_470WAY = '../../data/phylop/hg38.phyloP470way.bw'

OUTPUT_DIR = Path('../results/phylop')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FLANK_SIZE = 150  # bases of EXONIC sequence

## Load Transcript Exon Structures from GENCODE

In [ ]:
print("Loading transcript exon structures from GENCODE GTF...")

transcript_exons = defaultdict(list)

with gzip.open(GENCODE_GTF, 'rt') as f:
    for line in f:
        if line.startswith('#'):
            continue
        
        fields = line.strip().split('\t')
        if len(fields) < 9 or fields[2] != 'exon':
            continue
        
        chrom = fields[0]
        start = int(fields[3]) - 1  # GTF is 1-based, convert to 0-based
        end = int(fields[4])
        strand = fields[6]
        
        # Parse attributes for transcript_id
        attrs = {}
        for attr in fields[8].split(';'):
            attr = attr.strip()
            if attr:
                parts = attr.split(' ', 1)
                if len(parts) == 2:
                    key, val = parts
                    attrs[key] = val.strip('"')
        
        transcript_id = attrs.get('transcript_id', '')
        if not transcript_id:
            continue
        
        transcript_exons[transcript_id].append({
            'chrom': chrom,
            'start': start,
            'end': end,
            'strand': strand
        })

# Sort exons by start position
for transcript_id in transcript_exons:
    transcript_exons[transcript_id].sort(key=lambda x: x['start'])

print(f"Loaded exons for {len(transcript_exons):,} transcripts")

## Load Translon Data

In [ ]:
# Load v2 results (has block structure)
print(f"Loading v2 results from {V2_RESULTS}...")
v2_df = pd.read_csv(V2_RESULTS, sep='\t')
v2_df = v2_df[v2_df['phylop_dataset'] == '470way'].copy()
print(f"Loaded {len(v2_df):,} translons (470way)")

# Load transcript annotations
print(f"Loading transcript annotations from {TRANSCRIPT_ANNOTATIONS}...")
transcript_df = pd.read_csv(TRANSCRIPT_ANNOTATIONS, sep='\t')
print(f"Loaded {len(transcript_df):,} translon-transcript mappings")

# Merge
translons = v2_df.merge(
    transcript_df[['orf_name', 'transcript']],
    left_on='translon_id',
    right_on='orf_name',
    how='left'
)

print(f"\nMerged dataset:")
print(f"  Total translons: {len(translons):,}")
print(f"  With transcript ID: {translons['transcript'].notna().sum():,}")
print(f"  Without transcript ID: {translons['transcript'].isna().sum():,}")

translons.head()

## Helper Functions

In [ ]:
def parse_bed_blocks(chrom_start, block_count, block_sizes_str, block_starts_str):
    """
    Parse BED12 block notation into list of (start, end) tuples.
    """
    if pd.isna(block_sizes_str) or pd.isna(block_starts_str):
        return []
    
    try:
        block_sizes = [int(x) for x in str(block_sizes_str).rstrip(',').split(',')]
        block_starts = [int(x) for x in str(block_starts_str).rstrip(',').split(',')]
        
        blocks = []
        for i in range(int(block_count)):
            block_start = chrom_start + block_starts[i]
            block_end = block_start + block_sizes[i]
            blocks.append((block_start, block_end))
        
        return blocks
    except:
        return []

In [ ]:
def get_exonic_flanks(translon_chrom, translon_blocks, translon_strand, transcript_id, flank_size=150):
    """
    Get exonic flanking regions by walking along transcript exons.
    
    Returns:
        upstream_coords: list of (start, end) tuples
        downstream_coords: list of (start, end) tuples
    """
    if not translon_blocks:
        return [], []
    
    translon_min = min(s for s, e in translon_blocks)
    translon_max = max(e for s, e in translon_blocks)
    
    # Get transcript exons
    exons = transcript_exons.get(transcript_id, [])
    
    # Fallback to genomic flanks if no transcript annotation
    if not exons or pd.isna(transcript_id):
        if translon_strand == '+':
            upstream_coords = [(max(0, translon_min - flank_size), translon_min)]
            downstream_coords = [(translon_max, translon_max + flank_size)]
        else:
            upstream_coords = [(translon_max, translon_max + flank_size)]
            downstream_coords = [(max(0, translon_min - flank_size), translon_min)]
        return upstream_coords, downstream_coords
    
    # Separate exons: before, overlapping, after translon
    before_exons = [e for e in exons if e['end'] <= translon_min]
    after_exons = [e for e in exons if e['start'] >= translon_max]
    translon_exons = [e for e in exons if not (e['end'] <= translon_min or e['start'] >= translon_max)]
    
    strand = exons[0]['strand']
    
    if strand == '+':
        # Upstream: walk backwards collecting exonic bases
        upstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases before translon
        for exon in reversed(translon_exons):
            if remaining <= 0:
                break
            if exon['start'] < translon_min:
                take_start = max(exon['start'], translon_min - remaining)
                take_end = translon_min
                upstream_coords.insert(0, (take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk backwards through before_exons
        for exon in reversed(before_exons):
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                upstream_coords.insert(0, (exon['start'], exon['end']))
                remaining -= exon_len
            else:
                upstream_coords.insert(0, (exon['end'] - remaining, exon['end']))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_start = min(e['start'] for e in exons)
            upstream_coords.insert(0, (max(0, transcript_start - remaining), transcript_start))
        
        # Downstream: walk forwards
        downstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases after translon
        for exon in translon_exons:
            if remaining <= 0:
                break
            if exon['end'] > translon_max:
                take_start = translon_max
                take_end = min(exon['end'], translon_max + remaining)
                downstream_coords.append((take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk forwards through after_exons
        for exon in after_exons:
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                downstream_coords.append((exon['start'], exon['end']))
                remaining -= exon_len
            else:
                downstream_coords.append((exon['start'], exon['start'] + remaining))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_end = max(e['end'] for e in exons)
            downstream_coords.append((transcript_end, transcript_end + remaining))
    
    else:  # strand == '-'
        # For minus strand: upstream (5' of gene) is genomically downstream
        upstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases after translon (genomically)
        for exon in translon_exons:
            if remaining <= 0:
                break
            if exon['end'] > translon_max:
                take_start = translon_max
                take_end = min(exon['end'], translon_max + remaining)
                upstream_coords.append((take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk through after_exons
        for exon in after_exons:
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                upstream_coords.append((exon['start'], exon['end']))
                remaining -= exon_len
            else:
                upstream_coords.append((exon['start'], exon['start'] + remaining))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_end = max(e['end'] for e in exons)
            upstream_coords.append((transcript_end, transcript_end + remaining))
        
        # Downstream (3' of gene) = genomically upstream
        downstream_coords = []
        remaining = flank_size
        
        # Check translon exons for bases before translon
        for exon in reversed(translon_exons):
            if remaining <= 0:
                break
            if exon['start'] < translon_min:
                take_start = max(exon['start'], translon_min - remaining)
                take_end = translon_min
                downstream_coords.insert(0, (take_start, take_end))
                remaining -= (take_end - take_start)
        
        # Walk backwards through before_exons
        for exon in reversed(before_exons):
            if remaining <= 0:
                break
            exon_len = exon['end'] - exon['start']
            if exon_len <= remaining:
                downstream_coords.insert(0, (exon['start'], exon['end']))
                remaining -= exon_len
            else:
                downstream_coords.insert(0, (exon['end'] - remaining, exon['end']))
                remaining = 0
        
        # Extend genomically if needed
        if remaining > 0:
            transcript_start = min(e['start'] for e in exons)
            downstream_coords.insert(0, (max(0, transcript_start - remaining), transcript_start))
    
    return upstream_coords, downstream_coords

In [ ]:
def extract_phylop_scores(bw, chrom, coord_list):
    """Extract PhyloP scores from multiple coordinate ranges."""
    all_scores = []
    
    for start, end in coord_list:
        scores = bw.values(chrom, start, end, numpy=True)
        if scores is not None:
            valid_scores = scores[~np.isnan(scores)]
            if len(valid_scores) > 0:
                all_scores.extend(valid_scores)
    
    return np.array(all_scores) if all_scores else np.array([])


def calc_stats(scores):
    """Calculate statistics for PhyloP scores."""
    if len(scores) == 0:
        return {
            'n_bases': 0,
            'mean': np.nan,
            'median': np.nan,
            'std': np.nan,
            'min': np.nan,
            'max': np.nan
        }
    return {
        'n_bases': len(scores),
        'mean': np.mean(scores),
        'median': np.median(scores),
        'std': np.std(scores),
        'min': np.min(scores),
        'max': np.max(scores)
    }

## Process PhyloP Data with Exonic Flanks

In [ ]:
print(f"Opening PhyloP 470way: {PHYLOP_470WAY}")
bw = pyBigWig.open(PHYLOP_470WAY)

results = []

for idx, row in translons.iterrows():
    if idx % 1000 == 0:
        print(f"Processing {idx+1:,}/{len(translons):,}...")
    
    translon_id = row['translon_id']
    chrom = row['chrom']
    start = row['start']
    end = row['end']
    strand = row['strand']
    transcript_id = row.get('transcript', None)
    
    # Parse translon blocks
    block_count = row.get('blockCount', 1)
    block_sizes = row.get('blockSizes', '')
    block_starts = row.get('blockStarts', '')
    
    translon_blocks = parse_bed_blocks(start, block_count, block_sizes, block_starts)
    if not translon_blocks:
        translon_blocks = [(start, end)]
    
    # Get exonic flanks
    upstream_coords, downstream_coords = get_exonic_flanks(
        chrom, translon_blocks, strand, transcript_id, FLANK_SIZE
    )
    
    # Extract PhyloP scores
    feature_scores = extract_phylop_scores(bw, chrom, translon_blocks)
    upstream_scores = extract_phylop_scores(bw, chrom, upstream_coords)
    downstream_scores = extract_phylop_scores(bw, chrom, downstream_coords)
    
    # Calculate statistics
    feature_stats = calc_stats(feature_scores)
    upstream_stats = calc_stats(upstream_scores)
    downstream_stats = calc_stats(downstream_scores)
    
    # Derived metrics
    feature_mean = feature_stats['mean']
    upstream_mean = upstream_stats['mean']
    downstream_mean = downstream_stats['mean']
    
    if not np.isnan(feature_mean) and not np.isnan(upstream_mean) and not np.isnan(downstream_mean):
        conservation_specificity = feature_mean - max(upstream_mean, downstream_mean)
    else:
        conservation_specificity = np.nan
    
    # Store result
    result = {
        'translon_id': translon_id,
        'chrom': chrom,
        'start': start,
        'end': end,
        'strand': strand,
        'transcript_id': transcript_id if pd.notna(transcript_id) else 'no_annotation',
        'exonic_length': row['exonic_length'],
        'blockCount': block_count,
        'flank_size': FLANK_SIZE,
        'flank_type': 'exonic',
        
        'feature_n_bases': feature_stats['n_bases'],
        'feature_mean': feature_stats['mean'],
        'feature_median': feature_stats['median'],
        'feature_std': feature_stats['std'],
        'feature_min': feature_stats['min'],
        'feature_max': feature_stats['max'],
        
        'upstream_n_bases': upstream_stats['n_bases'],
        'upstream_mean': upstream_stats['mean'],
        'upstream_median': upstream_stats['median'],
        'upstream_std': upstream_stats['std'],
        'upstream_min': upstream_stats['min'],
        'upstream_max': upstream_stats['max'],
        
        'downstream_n_bases': downstream_stats['n_bases'],
        'downstream_mean': downstream_stats['mean'],
        'downstream_median': downstream_stats['median'],
        'downstream_std': downstream_stats['std'],
        'downstream_min': downstream_stats['min'],
        'downstream_max': downstream_stats['max'],
        
        'conservation_specificity': conservation_specificity,
    }
    
    results.append(result)

bw.close()

print(f"\nCompleted processing {len(results):,} translons")

## Save Results

In [ ]:
df_results = pd.DataFrame(results)

output_file = OUTPUT_DIR / 'translon_phylop_v3_exonic_flanks.tsv'
df_results.to_csv(output_file, sep='\t', index=False)

print(f"Results saved to: {output_file}")
print(f"Total translons: {len(df_results):,}")
print(f"  With transcript annotation: {(df_results['transcript_id'] != 'no_annotation').sum():,}")
print(f"  Without transcript annotation: {(df_results['transcript_id'] == 'no_annotation').sum():,}")

## Quick Summary Statistics

In [ ]:
print("="*80)
print("V3 EXONIC FLANKS - SUMMARY")
print("="*80)
print(f"\nMean PhyloP scores:")
print(f"  Feature:    {df_results['feature_mean'].mean():6.3f}")
print(f"  Upstream:   {df_results['upstream_mean'].mean():6.3f}")
print(f"  Downstream: {df_results['downstream_mean'].mean():6.3f}")
print(f"\nConservation specificity: {df_results['conservation_specificity'].mean():6.3f}")

print(f"\nFlank base counts (should be ~{FLANK_SIZE}):")
print(f"  Upstream mean:   {df_results['upstream_n_bases'].mean():.1f}")
print(f"  Downstream mean: {df_results['downstream_n_bases'].mean():.1f}")

## Compare v2 (genomic) vs v3 (exonic) Flanks

In [ ]:
# Load v2 for comparison
v2_comparison = v2_df[['translon_id', 'upstream_mean', 'downstream_mean', 'conservation_specificity']].copy()
v2_comparison.columns = ['translon_id', 'v2_upstream', 'v2_downstream', 'v2_specificity']

# Merge with v3
comparison = df_results[['translon_id', 'upstream_mean', 'downstream_mean', 'conservation_specificity']].merge(
    v2_comparison, on='translon_id', how='left'
)
comparison.columns = ['translon_id', 'v3_upstream', 'v3_downstream', 'v3_specificity', 
                      'v2_upstream', 'v2_downstream', 'v2_specificity']

# Calculate differences
comparison['upstream_diff'] = comparison['v3_upstream'] - comparison['v2_upstream']
comparison['downstream_diff'] = comparison['v3_downstream'] - comparison['v2_downstream']
comparison['specificity_diff'] = comparison['v3_specificity'] - comparison['v2_specificity']

print("="*80)
print("COMPARISON: v2 (genomic) vs v3 (exonic) flanks")
print("="*80)
print(f"\nMean differences (v3 - v2):")
print(f"  Upstream:   {comparison['upstream_diff'].mean():+6.3f}")
print(f"  Downstream: {comparison['downstream_diff'].mean():+6.3f}")
print(f"  Specificity: {comparison['specificity_diff'].mean():+6.3f}")

print(f"\nIf v3 flanks are HIGHER than v2:")
print(f"  → Genomic flanks were hitting introns (low conservation)")
print(f"  → Exonic flanks show true exonic conservation")
print(f"  → Fewer translons will appear to have 'restricted' conservation")

comparison.head(10)